# Phase 2 — RNA-FM embedding extraction

**Purpose**: Run `test_set_sequences.fa` through RNA-FM and write `benchmark/results/embeddings/rna_fm.npz`.

- **GPU**: T4 (free tier OK)
- **Runtime setting**: Runtime → Change runtime type → GPU (T4), no need for Pro
- **Notes**: 100M parameters. ~10 min for 100 sequences on free Colab T4.

This notebook assumes Phase 1 has already produced `data/processed/test_set_sequences.fa` and that the repository is available on Drive at the path set in the next cell (default: `/content/drive/MyDrive/rna-foundation-grounding-benchmark`).


## 1. Drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
REPO = os.environ.get('REPO_PATH', '/content/drive/MyDrive/rna-foundation-grounding-benchmark')
assert os.path.isdir(REPO), f'Repo not found at {REPO}. Upload or adjust REPO_PATH.'
os.chdir(REPO)
print('CWD:', os.getcwd())


## 2. Install dependencies

In [ ]:
!pip install -q numpy pandas scipy scikit-learn torch biopython
!pip install -q multimolecule>=0.0.5 transformers>=4.36


## 3. Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Memory (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)


## 4. Verify input exists

In [ ]:
FA = f'{REPO}/data/processed/test_set_sequences.fa'
assert os.path.isfile(FA), f'Missing {FA}. Run Phase 1 first.'
n = 0
with open(FA) as f:
    for line in f:
        if line.startswith('>'): n += 1
print(f'{n} sequences found in test_set_sequences.fa')


## 5. Run RNA-FM extraction

In [ ]:
import sys
sys.path.insert(0, f'{REPO}/benchmark')
!python {REPO}/benchmark/models/rna_fm.py


## 6. Verify output

In [ ]:
import numpy as np
OUT = f'{REPO}/benchmark/results/embeddings/rna_fm.npz'
assert os.path.isfile(OUT), f'Extraction failed: {OUT} not found'
z = np.load(OUT, allow_pickle=True)
print('keys:', list(z.keys()))
print('n_genes:', len(z['gene_ids']))
print('embedding shape:', z['embeddings'].shape)


## 7. Download to local (optional)

The output is already on Drive; this cell is only for pulling a copy straight to your local machine.


In [ ]:
from google.colab import files
files.download(f'{REPO}/benchmark/results/embeddings/rna_fm.npz')


## Done — next step

Once `rna_fm.npz` is generated, either move on to the next model notebook, or — once all five are present — run `python benchmark/eval.py` locally.
